# all-reduce-compose — ex1: compose all_reduce from reduce plus broadcast

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `all-reduce-compose`. Running the final beacon cell reports progress against the `Distributed: all_reduce composition` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce composition` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-compose`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-compose"
DD_SUBTOPIC = "Distributed: all_reduce composition"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.distributed quick refresher

PyTorch's collective-communication library (`torch.distributed`, aliased `dist`) lets multiple processes coordinate over tensors. The standard workflow:

1. **Each rank** runs the same function, parameterized by `rank` and `world_size`. Rank 0 is conventionally the 'driver'.
2. **`dist.init_process_group(backend=...)`** establishes the rendezvous. Backends:
   - `'nccl'` — NVIDIA's GPU-to-GPU primitive. Used in ARENA's multi-GPU setup. Requires CUDA + one process per GPU.
   - `'gloo'` — CPU-friendly. What you'll use in these drills (Colab CPU runtimes have no real GPUs).
3. **Pin a device** per rank: `torch.device(f'cuda:{rank}')` so each process owns exactly one GPU.
4. **Collective ops** (`all_reduce`, `broadcast`, `send`, `recv`) operate in-place on tensors of identical shape across all ranks.
5. **`dist.destroy_process_group()`** tears down at the end.

**Two ways to launch multiple ranks:**
- `torch.multiprocessing.spawn(fn, args=(...), nprocs=world_size)` — what ARENA uses. Spawn requires the worker fn be importable (not defined in `__main__`/a notebook cell).
- `mp.get_context('fork').Process(target=fn, args=...)` — Linux-only but works with cell-defined fns. The drills use this in tests so the worker can stay in the cell.

**Two-rank trick.** Colab gives ~2 CPU cores, so `world_size=2` is the right scale: enough to exercise the protocol, cheap enough to finish in seconds.

### This drill's atom: `all_reduce = reduce ∘ broadcast`
ARENA's pedagogy: build `all_reduce` on top of the two primitives you already implemented:
```python
def all_reduce(tensor, rank, world_size, op='sum'):
    reduce(tensor, rank, world_size, dst=0, op=op)   # → rank 0 holds the result
    broadcast(tensor, rank, world_size, src=0)       # → every rank gets it
```
After `reduce`, only rank 0's tensor is correct. After the `broadcast`, every rank holds the same final value. **Real `dist.all_reduce`** does both in a single tree-reduction pass (faster, less memory) — but the composed form is the easy-to-reason-about correct baseline.

### Exercise 1 — compose all_reduce from reduce plus broadcast

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the `reduce`-then-`broadcast` composition to build a custom `all_reduce` that sums tensors across ranks, using real `dist.reduce` and `dist.broadcast` on the `gloo` backend.
> Keywords: all_reduce, reduce, broadcast, composition, tree-pattern
> ```

**KCs targeted:** `all-reduce-equals-reduce-then-broadcast`, `rank0-as-aggregation-point`

Implement `ex1_all_reduce_worker(rank, world_size, port, initial, out_queue)`. Each rank starts with `initial * (rank + 1)` and the composed `all_reduce` should leave EVERY rank holding the sum `initial * (1 + 2 + ... + world_size)`.

Steps inside the worker:
1. Init `gloo` process group.
2. Build `tensor = t.tensor([initial * (rank + 1)], dtype=t.float32)`.
3. **Compose `all_reduce` from `reduce` + `broadcast`:**
   ```python
   dist.reduce(tensor, dst=0, op=dist.ReduceOp.SUM)
   dist.broadcast(tensor, src=0)
   ```
   (We're using the REAL `dist.reduce` and `dist.broadcast` — this drill is about the composition pattern, not reimplementing send/recv. ARENA's hand-rolled versions live in the previous drills.)
4. Push `(rank, tensor.item())` onto `out_queue`.
5. Destroy process group.

**Check.** With `initial=1.0, world_size=3`, every rank should end with `6.0` (= 1+2+3). The test asserts this for ranks 0, 1, 2.

In [ ]:
import os, datetime
import torch as t
import torch.distributed as dist

def ex1_all_reduce_worker(rank, world_size, port, initial, out_queue):
    """Init gloo, compose all_reduce from reduce+broadcast, queue result."""
    raise NotImplementedError()


def _test_ex1():
    import os as _os
    import datetime as _dt
    import torch.distributed as _dist
    import torch.multiprocessing as _mp

    def _dd_run_workers(worker_fn, world_size, port, *extra_args, timeout=30):
        """Spawn `world_size` fork-context procs, return list of exitcodes."""
        ctx = _mp.get_context('fork')
        procs = []
        for rank in range(world_size):
            p = ctx.Process(target=worker_fn, args=(rank, world_size, port, *extra_args))
            p.start()
            procs.append(p)
        for p in procs:
            p.join(timeout=timeout)
        codes = [p.exitcode for p in procs]
        for p in procs:
            if p.is_alive():
                p.terminate()
        return codes

    manager = _mp.Manager()
    q = manager.Queue()
    codes = _dd_run_workers(ex1_all_reduce_worker, 3, 29540, 1.0, q)
    assert codes == [0, 0, 0], f'workers failed: {codes}'

    results = {}
    while not q.empty():
        rank, val = q.get()
        results[rank] = val
    assert set(results.keys()) == {0, 1, 2}
    expected = 1.0 + 2.0 + 3.0
    for rank, val in results.items():
        assert abs(val - expected) < 1e-5, f'rank {rank}: got {val}, expected {expected}'

    # 2-rank case with different initial.
    q2 = manager.Queue()
    codes2 = _dd_run_workers(ex1_all_reduce_worker, 2, 29541, 2.5, q2)
    assert codes2 == [0, 0]
    results2 = {}
    while not q2.empty():
        rank, val = q2.get()
        results2[rank] = val
    expected2 = 2.5 * 1 + 2.5 * 2  # = 7.5
    for rank, val in results2.items():
        assert abs(val - expected2) < 1e-5, f'2-rank case rank {rank}: got {val}, expected {expected2}'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_all_reduce_worker(rank, world_size, port, initial, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size,
                            timeout=datetime.timedelta(seconds=20))
    tensor = t.tensor([initial * (rank + 1)], dtype=t.float32)
    # Compose: reduce → rank 0, then broadcast back to all ranks.
    dist.reduce(tensor, dst=0, op=dist.ReduceOp.SUM)
    dist.broadcast(tensor, src=0)
    out_queue.put((rank, tensor.item()))
    dist.destroy_process_group()
```

**Why the composition matters pedagogically.** Real DDP libs use `dist.all_reduce` directly — but understanding it as `reduce ∘ broadcast` explains:
- Why the result is identical on every rank (the broadcast step).
- Why summing across ranks needs O(world_size) bandwidth (each rank's tensor must visit rank 0 once, then return).
- Why mean is just `sum / world_size` after the all_reduce (the next drill).

**Tree reduction beats this.** `NCCL`'s `all_reduce` uses a ring-allreduce pattern: each chunk of the tensor flows around the ring of GPUs, getting summed pairwise. Total time is `2 * (N-1) * chunk_size / bandwidth`, vs `2 * N * tensor_size` for the naive compose. For large tensors on many GPUs, the speedup is enormous.

**The `dist.ReduceOp` enum.** Other ops: `SUM`, `PRODUCT`, `MAX`, `MIN`, `BAND`, `BOR`, `BXOR`, `PREMUL_SUM`. There's NO `MEAN` op — you always sum then divide (next drill).
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()